# PHASE 1: AUTO PROFILING & DIAGNOSTICS
## Phân tích phân phối và phát hiện Outliers

**Mục tiêu:**
- Phân tích phân phối các cột numeric
- Phát hiện missing values
- Phát hiện outliers (IQR method)
- Tạo báo cáo diagnostic và visualizations
- Tạo bản nháp config cho PHASE 2 (dựa trên kết quả diagnostics)

**Output files:**
- `outputs/diagnostic_report.csv` - Báo cáo phân phối
- `outputs/diagnostic_report.json` - Báo cáo chi tiết
- `outputs/boxplots.png` - Boxplots cho outlier detection
- `outputs/histograms.png` - Histograms cho distribution analysis
- `outputs/draft_config.yaml` - Bản nháp config

In [ ]:
import sys
import os
from pathlib import Path

# Add modules to path
sys.path.insert(0, str(Path.cwd() / 'modules'))

import pandas as pd
import numpy as np
import logging
import yaml
import json
from datetime import datetime

from diagnostics import DataDiagnostics
from config_handler import ConfigHandler, TransformConfig, OutlierConfig, ScalingConfig, ImputationConfig

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Modules imported successfully")

## Step 1: Load merged data từ PHASE 0

In [ ]:
input_path = './outputs/dataset_merged.csv'

if not os.path.exists(input_path):
    raise FileNotFoundError(f"Merged dataset not found. Run PHASE 0 first: {input_path}")

df = pd.read_csv(input_path)

print(f"✓ Loaded {input_path}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")

## Step 2: Initialize Diagnostics

In [ ]:
# Cấu hình diagnostics
diagnostics = DataDiagnostics(df)

# Chạy diagnostics với IQR method
diagnostic_report = diagnostics.run_diagnostics(
    outlier_method='iqr',
    iqr_multiplier=1.5
)

print("\n" + "="*50)
print("DIAGNOSTICS COMPLETE")
print("="*50)

## Step 3: Inspect diagnostic results

In [ ]:
# Summary
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)
missing = diagnostic_report.get('missing_analysis', {})
print(f"Total missing: {missing.get('total_missing')} / {missing.get('total_records')}")
print(f"Missing percentage: {missing.get('missing_percentage')}%")

if missing.get('columns_missing'):
    print("\nTop columns with missing values:")
    for col, info in sorted(missing['columns_missing'].items(), 
                           key=lambda x: x[1]['percentage'], 
                           reverse=True)[:5]:
        print(f"  {col}: {info['percentage']}%")

In [ ]:
# Distribution Analysis
print("\n" + "="*50)
print("DISTRIBUTION ANALYSIS (Sample)")
print("="*50)

dist = diagnostic_report.get('distribution_analysis', {})
sample_cols = list(dist.keys())[:3]  # Show first 3 columns

for col in sample_cols:
    stats = dist[col]
    print(f"\n{col}:")
    print(f"  Mean: {stats['mean']:.4f}, Median: {stats['median']:.4f}")
    print(f"  Min: {stats['min']:.4f}, Max: {stats['max']:.4f}")
    print(f"  Std: {stats['std']:.4f}, Skewness: {stats['skewness']:.4f}")

In [ ]:
# Outlier Detection
print("\n" + "="*50)
print("OUTLIER DETECTION (IQR)")
print("="*50)

outlier_info = diagnostic_report.get('outlier_detection', {})
total_outlier_records = outlier_info.get('total_outlier_records', 0)
outliers = outlier_info.get('outliers', {})

print(f"Total outlier records: {total_outlier_records}")
print(f"Columns with outliers: {len(outliers)}")

if outliers:
    print("\nTop columns by outlier count:")
    for col, info in sorted(outliers.items(), 
                           key=lambda x: x[1]['count'], 
                           reverse=True)[:5]:
        print(f"  {col}: {info['count']} outliers")
        print(f"    Bounds: [{info['bounds']['lower']:.4f}, {info['bounds']['upper']:.4f}]")

## Step 4: Save diagnostic reports

In [ ]:
output_folder = Path('./outputs')
output_folder.mkdir(exist_ok=True)

# Save CSV report
diagnostics.save_report_csv(str(output_folder / 'diagnostic_report.csv'))

# Save JSON report
diagnostics.save_report_json(str(output_folder / 'diagnostic_report.json'))

print("✓ Reports saved")

## Step 5: Create visualizations

In [ ]:
# Create visualizations
print("Creating visualizations...")
diagnostics.create_visualizations(
    output_folder=str(output_folder),
    figsize=(15, 12)
)

print("✓ Visualizations saved")

## Step 6: Create draft configuration

In [ ]:
# Create draft config based on diagnostics
config_handler = ConfigHandler()
draft_config = config_handler.create_default_config()

# Update with recommendations based on diagnostics
# Example: Adjust outlier multiplier if many outliers found
total_outliers = outlier_info.get('total_outlier_records', 0)
total_records = len(df)
outlier_percentage = (total_outliers / total_records * 100) if total_records > 0 else 0

print(f"Outlier percentage: {outlier_percentage:.2f}%")

if outlier_percentage > 5:
    print("  → Recommending higher IQR multiplier (1.8) due to high outlier percentage")
    draft_config['phase1']['outlier_detection']['iqr_multiplier'] = 1.8
else:
    print("  → Using default IQR multiplier (1.5)")

# Check for highly skewed distributions
skewed_cols = [col for col, stats in dist.items() if abs(stats['skewness']) > 2]
if skewed_cols:
    print(f"  → Found {len(skewed_cols)} highly skewed columns")
    print(f"  → Recommending LOG TRANSFORM")
    draft_config['phase1']['log_transform']['enabled'] = True

# Missing value strategy
missing_pct = missing.get('missing_percentage', 0)
if missing_pct > 10:
    print(f"  → High missing percentage ({missing_pct}%)")
    print(f"  → Recommending KNN imputation with n_neighbors=3")
    draft_config['phase2']['imputation']['n_neighbors'] = 3

print("\n✓ Draft config created")

## Step 7: Save draft config

In [ ]:
draft_config_path = output_folder / 'draft_config.yaml'
config_handler.save_config(str(draft_config_path), draft_config)

print(f"✓ Draft config saved to {draft_config_path}")
print("\n" + "="*50)
print("PHASE 1.5: REVIEW & EDIT CONFIG")
print("="*50)
print("\n⚠️  IMPORTANT: Please review and edit the config file:")
print(f"  File: {draft_config_path}")
print("\nCheck the following:")
print("  1. Log transform settings (enabled, features)")
print("  2. Outlier detection method and parameters")
print("  3. Scaling method (standard, minmax, robust)")
print("  4. Imputation parameters (n_neighbors)")
print("\n→ Save as: outputs/final_config.yaml when ready")
print("→ Then proceed to 02_execution_pipeline.ipynb")

## Checkpoint: PHASE 1 Complete ✓

**Output files created:**
- `outputs/diagnostic_report.csv` - Distribution statistics
- `outputs/diagnostic_report.json` - Detailed diagnostic results
- `outputs/boxplots.png` - Outlier visualization
- `outputs/histograms.png` - Distribution visualization
- `outputs/draft_config.yaml` - Auto-generated configuration

**Next Step: PHASE 1.5 - Manual Configuration**
1. Review the draft_config.yaml
2. Edit any parameters as needed
3. Save as final_config.yaml
4. Proceed to 02_execution_pipeline.ipynb